In [1]:
import os
import shutil
import kaggle

In [2]:
pwd

'c:\\Users\\User\\Documents\\chest-cancer-classification\\research'

In [3]:
os.chdir("../")

In [4]:
pwd

'c:\\Users\\User\\Documents\\chest-cancer-classification'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir : Path
    source_url : str
    local_data_file : Path
    unzip_dir : Path

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
from cnnClassifier import logger


In [7]:
class ConfigurationManager:
    def __init__(self,config_filepath = CONFIG_FILE_PATH,params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def data_ingestion_config(self)-> DataIngestionConfig:
        data_ingestion_config = self.config.data_ingestion

        create_directories([data_ingestion_config.root_dir])

        return DataIngestionConfig(data_ingestion_config.root_dir, data_ingestion_config.source_url, data_ingestion_config.local_data_file, data_ingestion_config.unzip_dir)


In [9]:
import zipfile
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [10]:


class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        """
        Downloads the dataset from Kaggle using the kaggle package.
        Reads KAGGLE_USERNAME and KAGGLE_KEY from the .env file.
        """
        try:
            dataset_slug = self.config.source_url          # "masoudnickparvar/brain-tumor-mri-dataset"
            zip_local_data_file = self.config.local_data_file
            download_dir = os.path.dirname(zip_local_data_file)

            os.makedirs(download_dir, exist_ok=True)

            kaggle.api.authenticate()   # picks up KAGGLE_USERNAME / KAGGLE_KEY from env

            logger.info(f"Downloading Kaggle dataset: {dataset_slug}")
            kaggle.api.dataset_download_files(
                dataset_slug,
                path=download_dir,
                unzip=False,
                quiet=False
            )

            # kaggle saves as <dataset-name>.zip — rename to match config path
            dataset_name = dataset_slug.split("/")[-1]
            downloaded_zip = os.path.join(download_dir, f"{dataset_name}.zip")
            if os.path.exists(downloaded_zip) and downloaded_zip != str(zip_local_data_file):
                os.rename(downloaded_zip, zip_local_data_file)

            logger.info(f"Dataset downloaded to: {zip_local_data_file}")

        except Exception as e:
            raise e

    def extract_file(self):
        """Extracts the downloaded ZIP into the unzip directory."""
        try:
            unzip_path = self.config.unzip_dir
            os.makedirs(unzip_path, exist_ok=True)

            with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
                zip_ref.extractall(unzip_path)

            logger.info(f"Dataset extracted to: {unzip_path}")

        except Exception as e:
            raise e

In [11]:
try:
    config_manager = ConfigurationManager()
    data_ingestion_config = config_manager.data_ingestion_config()
    data_ingestion = DataIngestion(data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_file()
except Exception as e:
    raise e

[2026-06-02 11:03:31,293: INFO : common : line 35 : yaml file: config\config.yaml loaded successfully]
[2026-06-02 11:03:31,327: INFO : common : line 35 : yaml file: params.yaml loaded successfully]
[2026-06-02 11:03:31,331: INFO : common : line 56 : created directory: artifacts]
[2026-06-02 11:03:31,334: INFO : common : line 56 : created directory: artifacts/data_ingestion]
[2026-06-02 11:03:31,338: INFO : 1744444097 : line 19 : Downloading Kaggle dataset: masoudnickparvar/brain-tumor-mri-dataset]
Dataset URL: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset


100%|██████████| 157M/157M [00:00<00:00, 539MB/s] 


[2026-06-02 11:04:21,327: INFO : 1744444097 : line 33 : Dataset downloaded to: artifacts/data_ingestion/data.zip]


[2026-06-02 11:04:37,734: INFO : 1744444097 : line 47 : Dataset extracted to: artifacts/data_ingestion]
